# SMS Spam Collection Classifier


### Introduction
This project aims to build a classification model that is able to distinguish between spam and non-spam messages in SMS messages.

The dataset is obtained from Kaggle and created by UCI Machine Learning : [https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)


First, I import the necessary Python libraries/packages. Additional imports (e.g. scikit-learn modules) will be added as needed throughout the project.

In [1]:
import pandas as pd
import numpy as np

### Goal 1: Load and Explore the Data
Load the CSV file into a pandas DataFrame and explore its structure such as columns, shape, data types, missing values, and class distribution.

In [2]:
# Load CSV file into pandas DataFrame
df = pd.read_csv('spam.csv', encoding='latin-1') # Using 'latin-1' encoding as the dataset contains characters incompatible with UTF-8

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


Here, we can see a few columns, v1, v2, and some other unnamed columns. Now, I want to get rid of the unnamed columns as they provide no information, as well as rename columns v1 and v2 into something more meaningful.

v1 is renamed into "Classification" and v2 is renamed into "Content".

In [3]:
# Drop unnecessary columns
df = df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Rename columns to be meaningful
df = df.rename(columns={'v1':'Classification', 'v2':'Content'})
df.head()

,Classification,Content
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Next, I check the shape, data types, and missing values of the dataset.

In [5]:
print(df.shape)
print() 
print(df.dtypes)
print() 
print(df.isnull().sum())

(5572, 2)

Classification    object
Content           object
dtype: object

Classification    0
Content           0
dtype: int64


This dataset contains 5572 rows and 2 columns after removing irrelevant columns.

The datatypes for both columns are *object*, which is equivalent to string data.

Upon checking the count of missing values for each columns, the dataset appears to have all rows and columns filled, there are no empty values.

Next, I examine the distribution of spam vs ham to check for class imbalance.

In [6]:
df['Classification'].value_counts()

Classification
ham     4825
spam     747
Name: count, dtype: int64

This is an imbalanced dataset that is skewed towards a larger amount of non-spam data. Roughly 87% of these messages are intended for the recipient.

### Goal 2: Text Preprocessing, Feature Extraction, and Train/Test Split

Before building a model, the text data needs to be converted into numerical features. I will use scikit-learn's *TfidfVectorizer* to handle both basic text cleaning (lowercasing, stopword removal) and feature extraction (TF-IDF scores) in one step.

The data is split into training and test sets before fitting the TF-IDF vectorizer to avoid data leakage, ensuring the model is never exposed to patterns from the test set during training.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer


In [11]:
# Define X and y as features and labels respectively
X = df['Content']
y = df['Classification']

# Splits data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize vectorizer which normalizes words to lowercase and removes stopwords
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')

# Fit vectorizer on training data and transform into TF-IDF features
X_train_tfidf = vectorizer.fit_transform(X_train)

# Only transform test data using to avoid data leakage
X_test_tfidf = vectorizer.transform(X_test)


### Goal 3: Build and Evaluate a Classification Model

Train a Multinomial Naive Bayes classifier on the TF-IDF features and evaluate its performance on the test set. Since the dataset is imbalanced, evaluation will go beyond accuracy and include precision, recall, F1-score, and a confusion matrix to better understand how the model performs on both spam and ham classes.

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

In [28]:
model = MultinomialNB() # Initialise machine learning model

model.fit(X_train_tfidf, y_train) # Train model using training set after TF-IDF transformation

predictions = model.predict(X_test_tfidf) # Evaluate model

report = classification_report(y_test, predictions)

matrix = confusion_matrix(y_test, predictions)

print(f'{model}'[:-2] +' Results')
print(report)
print(matrix)



MultinomialNB Results
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115

[[966   0]
 [ 35 114]]


A Multinomial Naive Bayes model is initialised and trained on the TF-IDF transformed training set. Predictions are then made on the test set and evaluated using a classification report and confusion matrix. Here, the model is trained on 4,457 samples (80%) and evaluated on the remaining 1,115 test samples (20%).

**Results:**

The model achieves 97% overall accuracy, however this can be misleading due to the class imbalance. Looking at the individual class metrics:

- **Ham**: 0.97 precision and 1.00 recall, nearly all non-spam messages are correctly identified.
- **Spam**: 1.00 precision but only 0.77 recall, while every message flagged as spam is correct, the model misses 23% of actual spam messages (35 out of 149).

The model is conservative in its spam predictions as it avoids false positives at the cost of letting some spam through.

### Goal 4: Model Comparison

Now that a baseline result has been established with Multinomial Naive Bayes, I will train and evaluate additional classification models to determine which performs best on this dataset. The models being compared are Decision Tree, K-Nearest Neighbours (KNN), and Logistic Regression, all evaluated on the same test set using the same TF-IDF features.

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [34]:
models = [DecisionTreeClassifier(), KNeighborsClassifier(), LogisticRegression()]

for model in models:
    
    # Initialise machine learning model

    model.fit(X_train_tfidf, y_train) # Train model using training set after TF-IDF transformation

    predictions = model.predict(X_test_tfidf) # Evaluate model

    report = classification_report(y_test, predictions)

    matrix = confusion_matrix(y_test, predictions)

    print(f'{model}'[:-2] +' Results')
    print(report)
    print(matrix)
    print("=========================================================================")
    print()



DecisionTreeClassifier Results
              precision    recall  f1-score   support

         ham       0.98      0.98      0.98       966
        spam       0.87      0.84      0.85       149

    accuracy                           0.96      1115
   macro avg       0.92      0.91      0.92      1115
weighted avg       0.96      0.96      0.96      1115

[[947  19]
 [ 24 125]]

KNeighborsClassifier Results
              precision    recall  f1-score   support

         ham       0.91      1.00      0.95       966
        spam       1.00      0.34      0.51       149

    accuracy                           0.91      1115
   macro avg       0.95      0.67      0.73      1115
weighted avg       0.92      0.91      0.89      1115

[[966   0]
 [ 98  51]]

LogisticRegression Results
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.76      0.86       149

    accuracy                           0.97    

**Model Comparison Results:**

| Model | Accuracy | Spam Precision | Spam Recall | Spam F1 |
|---|---|---|---|---|
| Naive Bayes (baseline) | 0.97 | 1.00 | 0.77 | 0.87 |
| Decision Tree | 0.96 | 0.87 | 0.83 | 0.85 |
| KNN | 0.91 | 1.00 | 0.34 | 0.51 |
| Logistic Regression | 0.97 | 1.00 | 0.76 | 0.97 |

- **KNN** performed the weakest, only catching 34% of spam messages. This is expected as KNN tends to struggle with high-dimensional data like TF-IDF features.
- **Decision Tree** achieved the highest spam recall (0.83) but at the cost of precision, it incorrectly flagged 18 legitimate messages as spam.
- **Logistic Regression** performed almost identically to Naive Bayes, with high precision but similar recall limitations.
- **Naive Bayes** remains a strong baseline with a good balance of precision and recall, and zero false positives on ham.

Overall, the main weakness across models is **spam recall**, the models tend to be conservative and let some spam slip through. The next step will be to explore ways to improve this.